In [1]:
import os
import json
import numpy as np
from scipy.stats import t, norm

In [2]:
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n > 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n > 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [11]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
display(resumen)

{'_info': {'n': 10, 'nivel_confianza': '99%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.02 ± 0.01',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '2.47 ± 0.15', 'tratamiento': '66.66 ± 0.73'},
    'OR': {'espera': '0.81 ± 0.03', 'tratamiento': '12.97 ± 0.05'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '199.07 ± 0.78'}},
   'Hospital_2': {'ED': '0.01 ± 0.01',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '2.2 ± 0.15', 'tratamiento': '66.97 ± 0.95'},
    'OR': {'espera': '0.72 ± 0.07', 'tratamiento': '13.01 ± 0.04'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '196.52 ± 1.02'}},
   'Hospital_3': {'ED': '0.01 ± 0.01',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '1.65 ± 0.14', 'tratamiento': '63.68 ± 0.6'},
    'OR': {'espera': '0.5 ± 0.06', 'tratamiento': '12.94 ± 0.05'},
    'SDU_WARD': {'espera': '0.0 ± 0.0', 'tratamiento': '186.27 ± 1.1'}}},
  'promedio_por_hospital': {'Hospital_1': '260.17 ± 1.3',
  

In [ ]:
"""
"""